In [1]:
print("hi")

hi


In [1]:
import json
import re
import os
from typing import List
from PIL import Image
from pydantic import BaseModel
import pdfplumber
import torch
import cv2
import numpy as np
# from pipe_fn import pipe 

from transformers import pipeline
from output_utils import save_split_output

from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)
# =========================================================
# LOAD MODEL
# =========================================================

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)


# =========================
# CROP PT → REFERENCE
# =========================
def process_pdf_pt(pdf_path):
    claims = {}

    with pdfplumber.open(pdf_path) as pdf:
        for page_idx, page in enumerate(pdf.pages):
            words = page.extract_words()

            pt_positions = []
            ref_positions = []

            for w in words:
                txt = w["text"].strip()

                if txt.startswith("PT:"):
                    pt_positions.append(w["top"])

                elif txt.startswith("Totals:"):
                    ref_positions.append(w["bottom"])

            if not pt_positions or not ref_positions:
                continue

            # compute the payor ONCE per page — every PT block on this
            # page shares the same Claim Payor box
            page_claim_payor = extract_claim_payor_from_page(page)

            ref_idx = 0

            for i, pt_top in enumerate(pt_positions):

                while ref_idx < len(ref_positions) and ref_positions[ref_idx] <= pt_top:
                    ref_idx += 1

                if ref_idx >= len(ref_positions):
                    break

                ref_bottom = ref_positions[ref_idx]

                crop_top = max(0, pt_top - 15)   # 15 points above PT


                cropped = page.within_bbox((0, crop_top, page.width, ref_bottom))
                img = cropped.to_image(resolution=300).original

                key = f"page_{page_idx+1}block{i+1}"
                expected_rows = count_service_rows(page, pt_top, ref_bottom)
                patient_name = extract_patient_name(page)
                claims[key] = {
                    "image": img,
                    "expected_rows": expected_rows,
                    "patient_name": patient_name,
                    "claim_payor": page_claim_payor,
                    "page_number": page_idx + 1,
                }

                ref_idx += 1

    return claims


def save_images(claims, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    paths = []

    for k, v in claims.items():
        path = os.path.join(out_dir, f"{k}.png")

        v["image"].save(path)

        paths.append({
            "image_path": path,
            "expected_rows": v["expected_rows"],
            "patient_name": v["patient_name"],
            "claim_payor": v["claim_payor"],
            "page_number": v["page_number"],
        })

    return paths

# =========================
# TABLE ENHANCEMENT
# =========================
def make_table(image_path):
    img = cv2.imread(image_path)
    gray = cv2.imread(image_path, 0)

    _, thresh = cv2.threshold(gray, 150, 255, cv2.THRESH_BINARY_INV)

    sums = np.sum(thresh, axis=1)
    th = (thresh.shape[1] * 255) * 0.6
    lines = np.where(sums > th)[0]

    for l in lines:
        cv2.line(img, (0, l), (thresh.shape[1], l), (0, 0, 0), 1)

    return img


def convert_amounts_to_string(obj):

    amount_fields = {
        "billed_amount",
        "ppo_discount",
        "non_covered",
        "other_coverage",
        "deductible",
        "patient_responsibility",
        "paid_amount"
    }

    if isinstance(obj, dict):
        new_obj = {}

        for k, v in obj.items():
            if k in amount_fields:
                try:
                    # strip "$" and "," before parsing — the VLM doesn't
                    # always obey the prompt's "remove $" instruction, and
                    # float("$104.00") throws every time otherwise, which
                    # was silently blanking every amount field on save.
                    cleaned = str(v).replace("$", "").replace(",", "").strip()
                    new_obj[k] = f"{float(cleaned):.2f}" if cleaned else ""
                except Exception as e:
                    print(f"⚠ amount field '{k}' failed to convert — got {v!r} ({type(v).__name__}): {e}")
                    new_obj[k] = ""
            else:
                new_obj[k] = convert_amounts_to_string(v)

        return new_obj

    elif isinstance(obj, list):
        return [convert_amounts_to_string(i) for i in obj]

    else:
        return obj

def count_service_rows(page, region_top, region_bottom):

    words = page.extract_words()

    service_rows = set()

    for w in words:

        text = w["text"].strip()

        # find procedure code anywhere in the word
        if re.fullmatch(r"D\d{4}[:;,.]?", text):

            y = float(w["top"])

            if region_top <= y <= region_bottom:
                service_rows.add(round(y, 1))
    print(service_rows)
    return len(service_rows)



# =========================
# PROMPT
# =========================
def build_prompt(pdf_name):

    return f"""
You are extracting structured financial data from a dental EOB table image.

STRICT RULES:
Output ONLY valid JSON (no explanation).
Follow the schema EXACTLY.
Do NOT add extra fields.
Do NOT skip required fields.
If a value is missing → use "" for text and "" for numbers.
All monetary values MUST be string numbers (no $, no commas).
Extract rows ONLY from the table between "Date of Service" and "Totals".
Ignore any text outside the table.

HEADER EXTRACTION:
patient_name → value after "PT:"
dob → value after "PT. DOB:"

ROW EXTRACTION RULES:
Each row contains:
Date of Service | Procedure | Billed Amount | PPO Discount | Non Covered | Other Coverage | Deductible Co-Pays | Patient Resp. | Paid

service_code → ONLY the 5 characters from procedure (eg. D2330) if "()" appears ignore  this "()" and return only the 5 chars.
date_of_service → keep full string (e.g., 02/26/26 - 02/26/26) 


AMOUNT PARSING:
Remove "$" and "," then convert to string
If empty →  ""

TOTALS:
Extract from "Totals" row exactly

OUTPUT FORMAT (STRICT JSON):

{{

  "patient_name": {{
    "value": "",
    "confidence": 0.0
  }},

  "dob": {{
    "value": "",
    "confidence": 0.0
  }},

  "services": [

    {{

      "date_of_service": {{
        "value": "",
        "confidence": 0.0
      }},

      "service_code": {{
        "value": "",
        "confidence": 0.0
      }},

      "billed_amount": {{
        "value": "",
        "confidence": 0.0
      }},

      "ppo_discount": {{
        "value": "",
        "confidence": 0.0
      }},

      "non_covered": {{
        "value": "",
        "confidence": 0.0
      }},

      "other_coverage": {{
        "value": "",
        "confidence": 0.0
      }},

      "deductible": {{
        "value": "",
        "confidence": 0.0
      }},

      "patient_responsibility": {{
        "value": "",
        "confidence": 0.0
      }},

      "paid_amount": {{
        "value": "",
        "confidence": 0.0
      }}

    }}

  ],

  "totals": {{

    "billed_amount": {{
      "value": "",
      "confidence": 0.0
    }},

    "ppo_discount": {{
      "value": "",
      "confidence": 0.0
    }},

    "non_covered": {{
      "value": "",
      "confidence": 0.0
    }},

    "other_coverage": {{
      "value": "",
      "confidence": 0.0
    }},

    "deductible": {{
      "value": "",
      "confidence": 0.0
    }},

    "patient_responsibility": {{
      "value": "",
      "confidence": 0.0
    }},

    "paid_amount": {{
      "value": "",
      "confidence": 0.0
    }}

  }}

}}

 For every extracted field, return:
   - value
   - confidence
 
VALUE + CONFIDENCE RULES:
 
For every field return:
{{
  "value": "",
  "confidence": ""
}}
 
VALUE:
- "value" = the exact text/value visibly present in the specified location.
- Read ONLY from the exact cell/row/column requested.
- Copy exactly as printed; preserve "$" and formatting when visible.
- Never guess, infer, calculate, copy, shift, or use values from another row,
  column, table section, or Totals row.
- If the exact location is blank, missing, or has no clearly readable value:
  value = ""
 
CONFIDENCE:
- "confidence" = confidence that the extracted value is actually present
  in that exact location.
- Use a number from 0.0 to 1.0 based ONLY on visual evidence.
- 1.0 = clearly visible and certain.
- 0.8–0.99 = clearly visible with minor uncertainty.
- 0.5–0.79 = visible but difficult/ambiguous.
- 0.1–0.49 = very unclear.
- 0.0 = blank, missing, or no reliable visual evidence.
 
IMPORTANT:
Confidence is NOT confidence that the value is mathematically correct
or logically expected. It is ONLY confidence that the value shown in
"value" is what is visibly printed in the exact requested location.
 
If value = "":
confidence MUST = 0.0.
"""
def parse_amount(x):
    if x is None or x == "":
        return 0.0
    return float(
        str(x)
        .replace("$", "")
        .replace(",", "")
        .strip()
    )

def compute_totals_from_services(services):
    return {
        "billed_amount": round(sum(parse_amount(s.get("billed_amount", "")) for s in services), 2),
        "ppo_discount": round(sum(parse_amount(s.get("ppo_discount", "")) for s in services), 2),
        "non_covered": round(sum(parse_amount(s.get("non_covered", "")) for s in services), 2),
        "other_coverage": round(sum(parse_amount(s.get("other_coverage", "")) for s in services), 2),
        "deductible": round(sum(parse_amount(s.get("deductible", "")) for s in services), 2),
        #"patient_responsibility": round(sum(parse_amount(s.get("patient_responsibility", "")) for s in services), 2),
        "paid_amount": round(sum(parse_amount(s.get("paid_amount", "")) for s in services), 2),
    }


def validate_patient_totals(patient: dict, patient_name: str, expected_row_count: int):

    services = patient.get("services", [])
    totals   = patient.get("totals", {})

    field_names = list(compute_totals_from_services([]).keys())   # ADD
    total_fields = len(field_names) 

    if not services:
        field_errors = [                                            # ADD
            {"field": f, "computed": 0.0, "extracted": None}
            for f in field_names
        ]
        return False, "No services found", [{"error": "empty services"}] + field_errors, total_fields

    computed_totals = compute_totals_from_services(services)

    result_validation = ""
    errors = []
    has_error = False

    print(f"\n🔍 Validation for [{patient_name}]")
    print("-" * 80)

    # ===== FIELD VALIDATION =====
    for field, computed_value in computed_totals.items():

        extracted_value = round(parse_amount(totals.get(field, "")), 2)

        diff = round(computed_value - extracted_value, 2)
        match = abs(diff) <= 0.01   # 🔥 tolerance fix

        if match:
            icon = "✅"
            status = "MATCH"
        else:
            icon = "❌"
            status = "MISMATCH"
            has_error = True

            errors.append({
                "type": "field_mismatch",
                "field": field,
                "computed": computed_value,
                "extracted": extracted_value,
                "difference": diff
            })

        line = f"{icon} {field:25s} computed={computed_value:<10} | extracted={extracted_value:<10} {status}"
        print(line)
        result_validation += "\n" + line

    # ===== ROW COUNT VALIDATION =====
    extracted_row_count = len(services)

    if expected_row_count == extracted_row_count:
        icon = "✅"
        status = "MATCH"
    else:
        icon = "❌"
        status = "MISMATCH"
        has_error = True

        errors.append({
            "type": "row_count_mismatch",
            "expected_rows": expected_row_count,
            "extracted_rows": extracted_row_count
        })

    line = f"{icon} {'total_record_rows':25s} computed={expected_row_count:<10} | extracted={extracted_row_count:<10} {status}"
    print(line)
    result_validation += "\n" + line

    print("-" * 80)

    # ===== FINAL STATUS =====
    if has_error:
        print(f"❌ [{patient_name}] Validation FAILED\n")
        return False, result_validation, errors, total_fields
    else:
        print(f"✅ [{patient_name}] Validation PASSED\n")
        return True, result_validation, [], total_fields

def fix_patient_resp_equals_deductible(obj):
    """
    If patient_responsibility >= deductible:
        patient_responsibility = patient_responsibility - deductible

    The deductible value is NOT modified.
    Works recursively for nested dict/list structures.
    """

    if isinstance(obj, dict):

        if "patient_responsibility" in obj and "deductible" in obj:

            patient_str = re.sub(r"[^\d.]", "", str(obj.get("patient_responsibility", "")).strip())
            deductible_str = re.sub(r"[^\d.]", "", str(obj.get("deductible", "")).strip())

            try:
                patient = float(patient_str) if patient_str else 0.0
                deductible = float(deductible_str) if deductible_str else 0.0

                if patient >= deductible and deductible > 0:
                    obj["patient_responsibility"] = f"{patient - deductible:.2f}"

            except ValueError:
                pass

        for k, v in obj.items():
            obj[k] = fix_patient_resp_equals_deductible(v)

    elif isinstance(obj, list):
        obj = [fix_patient_resp_equals_deductible(i) for i in obj]

    return obj
# =========================
# JSON CLEANER
# =========================
def extract_json(text):
    start = text.find("{")
    end = text.rfind("}") + 1
    return json.loads(text[start:end])

def save_json(data, output_path):
    with open(output_path, "w", encoding = "utf-8") as f:
        json.dump(data, f, indent = 2, ensure_ascii = False)

# =========================
# MAIN PIPELINE
# =========================
# -----------------------------------------------------
# DENIAL DETECTION
# -----------------------------------------------------
def check_claim_denied(pdf_path):

    """
    Logic:
    Search ONLY before:
    'Attention Non-contracted Medicare Providers'

    If denied / denial keywords exist before that section,
    return True else False
    """

    denial_keywords=[
        "denied",
        "denial"
    ]
    stop_word="Explanation of Benefits(continued)"

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            full_text=page.extract_text()
            if not full_text:
                continue
            searchable_text =full_text.lower()

            # remove notes section
            for word in stop_word:
                if word in searchable_text:
                    searchable_text=searchable_text.split(word)[0]
                    break
            # search keywords
            for keyword in denial_keywords:
                if keyword in searchable_text:
                    print(f"claim denied keyword found:{keyword}")
                    return "denied"
        return "not denied"
#----------------Ftotal
# date cleaner
#----------------
def clean_date_of_service(obj):
    """
    Converts:
        10/14/24-10/14/24
        10/14/24 - 10/14/24
    into:
        10/14/24
    """

    if isinstance(obj, dict):
        new_obj = {}

        for k, v in obj.items():
            if k == "date_of_service" and isinstance(v, str):
                # split on '-' with optional spaces
                new_obj[k] = re.split(r"\s*-\s*", v.strip())[0]
            else:
                new_obj[k] = clean_date_of_service(v)

        return new_obj

    elif isinstance(obj, list):
        return [clean_date_of_service(i) for i in obj]

    return obj

import re
def _clean_field(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip()


def extract_patient_name(page):
    words = page.extract_words()
    if not words:
        return ""

    pt_anchors = [w for w in words if w["text"].strip().upper() == "PT:"]
    if not pt_anchors:
        return ""
    anchor = min(pt_anchors, key=lambda w: w["top"])  # topmost "PT:" on the page
    anchor_top = anchor["top"]
    anchor_x0 = anchor["x0"]

    # Horizontal bound: name column starts right after "PT:"; the next
    # column ("PT. DOB:") starts ~120pt further right, so cap comfortably
    # below that to keep multi-word wrapped lines without pulling in DOB.
    cap_x1 = anchor["x1"] + 110

    # Vertical bound: find the nearest other field label sharing PT:'s
    # left margin (e.g. "MBR:") above and below it. The midpoint between
    # PT:'s row and each neighbor's row is the natural split point between
    # this field's value and the next field's, even when both wrap.
    label_col = sorted(
        (
            w for w in words
            if abs(w["x0"] - anchor_x0) <= 3 and w["text"].strip().endswith(":")
        ),
        key=lambda w: w["top"],
    )
    next_label_top = next((w["top"] for w in label_col if w["top"] > anchor_top + 1), None)
    prev_label_top = next(
        (w["top"] for w in reversed(label_col) if w["top"] < anchor_top - 1), None
    )
    lower_limit = (anchor_top + next_label_top) / 2 if next_label_top else anchor_top + 40
    upper_limit = (prev_label_top + anchor_top) / 2 if prev_label_top else anchor_top - 40

    value_words = sorted(
        (
            w for w in words
            if upper_limit <= w["top"] <= lower_limit
            and anchor["x1"] < w["x0"] < cap_x1
        ),
        key=lambda w: (round(w["top"], 1), w["x0"]),
    )

    return _clean_field(" ".join(w["text"].strip() for w in value_words))


def extract_claim_payor_from_page(page):
    """
    Extract Claim Payor from a single page's Claim Payor box by reading
    ONLY the bold line(s) immediately following the 'Claim Payor:' label —
    works whether or not a P.O. Box line is present, since we stop at the
    first non-bold line instead of pattern-matching "P.O. Box".

    Returns:
        str : e.g. 'HillCour', 'Anthem Blue Cross and Blue Shield',
                    'Anthem Dental Claims', 'UMR'
              or "" if this page has no Claim Payor box.
    """

    words = page.extract_words(extra_attrs=["fontname"])

    anchor = None
    anchor_bottom = None
    for i, w in enumerate(words):
        if w["text"].strip().rstrip(":").lower() == "claim" and i + 1 < len(words):
            nxt = words[i + 1]
            if nxt["text"].strip().lower().startswith("payor"):
                anchor = w
                anchor_bottom = max(w["bottom"], nxt["bottom"])
                break

    if anchor is None:
        return ""

    x0 = anchor["x0"]

    # words below the label, roughly in the same column, within a
    # reasonable vertical window (avoids grabbing unrelated text further down)
    candidates = [
        w for w in words
        if w["top"] > anchor_bottom - 1
        and w["x0"] >= x0 - 5
        and w["top"] <= anchor_bottom + 60
    ]
    candidates.sort(key=lambda w: (round(w["top"], 1), w["x0"]))

    # group words into lines (same "top" within a small tolerance)
    lines = []
    current_line, current_top = [], None
    for w in candidates:
        if current_top is None or abs(w["top"] - current_top) <= 2:
            current_line.append(w)
            current_top = w["top"] if current_top is None else current_top
        else:
            lines.append(current_line)
            current_line, current_top = [w], w["top"]
    if current_line:
        lines.append(current_line)

    # keep consuming lines while they're bold; stop at first non-bold line
    payor_words = []
    for line in lines:
        is_bold = any("bold" in w.get("fontname", "").lower() for w in line)
        if not is_bold:
            break
        payor_words.extend(w["text"] for w in line)

    return " ".join(payor_words).strip()


def get_service_code_signature(services):
    """Ordered tuple of service codes, used to detect duplicate blocks."""
    return tuple(s.get("service_code", "").strip().upper() for s in services)


def dedupe_duplicate_patients(results):
    """
    Drops patient blocks that are duplicates of an earlier block —
    same patient_name AND the exact same sequence of service_codes.
    Keeps the FIRST occurrence, discards later ones.
    """
    seen = set()
    deduped = []

    for patient in results:
        name = patient.get("patient_name", "").strip().upper()
        sig = get_service_code_signature(patient.get("services", []))
        key = (name, sig)

        if key in seen:
            print(f"⚠ Skipping duplicate block for [{name}] services={sig}")
            continue

        seen.add(key)
        deduped.append(patient)

    return deduped

def clean_service_codes(obj):
    """
    Cleans service_code values by keeping only the valid
    5-character dental procedure code.

    Examples:
        D0150 ()  -> D0150
        D0274 ()  -> D0274
        D0220     -> D0220
        D4341 ()  -> D4341
    """

    if isinstance(obj, dict):
        new_obj = {}

        for k, v in obj.items():

            if k == "service_code" and isinstance(v, str):
                # Extract only D + 4 digits
                match = re.search(r"\bD\d{4}\b", v.upper())

                if match:
                    new_obj[k] = match.group(0)
                else:
                    new_obj[k] = v.strip()

            else:
                new_obj[k] = clean_service_codes(v)

        return new_obj

    elif isinstance(obj, list):
        return [clean_service_codes(item) for item in obj]

    return obj
    

def run_pipeline(pdf_path, output_dir="EOB_OUTPUT/Zelis", company_name = "Anthem_zelis"):

    pdf_full_name = os.path.basename(pdf_path)
    pdf_name = os.path.basename(pdf_path).split(".")[0].split("_")[-1]

    base_dir = os.path.join(output_dir, pdf_name)
    cropped_dir = os.path.join(base_dir, "cropped_images")

    os.makedirs(base_dir, exist_ok=True)
    os.makedirs(cropped_dir, exist_ok=True)

    claims = process_pdf_pt(pdf_path)
    image_paths = save_images(claims, cropped_dir)

    final_prompt = build_prompt(pdf_name)
    is_denied = check_claim_denied(pdf_path)
    # claim_payor = extract_claim_payor(pdf_path)
    # print(f"Claim Payor: {claim_payor}")
    print(f"claim status :{is_denied}")

    results = []

    for idx, item in enumerate(image_paths):
		
        img_path = item["image_path"]
        expected_rows = item["expected_rows"]

        print(f"Processing {idx+1}/{len(image_paths)}")

        image = make_table(img_path)
        image = Image.fromarray(image).convert("RGB")

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": image},
                    {"type": "text", "text": final_prompt}
                ]
            }
        ]

        with torch.no_grad():
            output = pipe(messages, max_new_tokens=2500, temperature= 0.0, do_sample=False)

        raw = output[0]["generated_text"]

        if isinstance(raw, list):
            raw = raw[-1]["content"]

        try:
            # parsed = extract_json(raw)
            # parsed["patient_name"] = item["patient_name"]

            # parsed["claim_payor"] = item["claim_payor"]   # <-- per-block payor
            # parsed["page_number"] = item["page_number"]

            parsed = extract_json(raw)

            model_confidence = calculate_model_confidence(parsed)   # ADD
            parsed = _unwrap_vlm_output(parsed) 

            parsed = {
                "patient_name": item["patient_name"],
                "claim_payor": item["claim_payor"],
                "dob": parsed.get("dob", ""),
                "services": parsed.get("services", []),
                "totals": parsed.get("totals", {}),
                "page_number": item["page_number"],
                "_model_confidence": model_confidence,   # ADD
            }

            parsed = convert_amounts_to_string(parsed)
            parsed = clean_date_of_service(parsed)
            parsed = fix_patient_resp_equals_deductible(parsed)
            parsed = clean_service_codes(parsed)


            results.append(parsed)
            parsed["_expected_rows"] = expected_rows
            print("✔ extracted")
        except Exception as e:
            print(f"❌ json failed:{e}")

        results = dedupe_duplicate_patients(results)

    for patient in results:

        is_valid, log, errors, total_fields  = validate_patient_totals(
            patient=patient,
            patient_name=patient.get("patient_name", ""),
            expected_row_count=patient.get("_expected_rows", 0)  # or your external expected count
        )

        patient["validation"] = {
            "status": is_valid,
            "errors": errors
        }

    confidence_results = list(results)                              # ADD
    confidence_score = calculate_eob_confidence(confidence_results)  # ADD

    for patient in results:                                          # ADD (cleanup)
        patient.pop("_expected_rows", None)
        patient.pop("_total_fields", None)
        patient.pop("_model_confidence", None)

    final =[
        {
        "eob_id": pdf_name,
        "file_name":pdf_full_name,
        # "claim_payor": claim_payor,        
        "claim_status": is_denied,
        "confidence_score": confidence_score,
        "patients": results
        }
    ]
    success_path, failed_path = save_split_output(
        final, company_name=company_name,
        pdf_name=pdf_name,
        pdf_path=pdf_path,
        cropped_dir=cropped_dir,
    )
    print(f"Cropped images saved in: {cropped_dir}")
    print(f"Success json: {success_path}")
    print(f"Failed json: {failed_path}")
    print(f"final response {final}")
    return final



    # json_output_path = os.path.join(base_dir, f"{pdf_name}_output.json")
    # save_json(final, json_output_path)

    # print(f"Cropped images saved in: {cropped_dir}")
    # print(f"Json output saved in: {json_output_path}")
    # print(f"final response {final}")
    # return final

W0901 17:59:15.075000 3368192 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 17:59:15.090000 3368192 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [ ]:
run_pipeline(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/ANTHEM/Anthem_PDF/Pmt_EOP_837531015.pdf")

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.


{355.5, 368.5, 404.1, 436.3, 342.4, 381.5}
claim status :not denied
Processing 1/1


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✔ extracted

🔍 Validation for [CHARLOTTE, NC 28277 JORDAN DEJACO]
--------------------------------------------------------------------------------
✅ billed_amount             computed=1054.0     | extracted=1054.0     MATCH
✅ ppo_discount              computed=383.0      | extracted=383.0      MATCH
✅ non_covered               computed=0.0        | extracted=0.0        MATCH
✅ other_coverage            computed=0.0        | extracted=0.0        MATCH
✅ deductible                computed=118.0      | extracted=118.0      MATCH
✅ paid_amount               computed=553.0      | extracted=553.0      MATCH
✅ total_record_rows         computed=6          | extracted=6          MATCH
--------------------------------------------------------------------------------
✅ [CHARLOTTE, NC 28277 JORDAN DEJACO] Validation PASSED

✅ Success output + pdf + crops saved: EOB_output_success/Anthem_zelis/837531015
Cropped images saved in: EOB_OUTPUT/Zelis/837531015/cropped_images
Success json: EOB_output_succ

[{'eob_id': '837531015',
  'file_name': 'Pmt_EOP_837531015.pdf',
  'claim_status': 'not denied',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'CHARLOTTE, NC 28277 JORDAN DEJACO',
    'claim_payor': 'UMR',
    'dob': '',
    'services': [{'date_of_service': '02/13/26',
      'service_code': 'D0150',
      'billed_amount': '139.00',
      'ppo_discount': '65.00',
      'non_covered': '0.00',
      'other_coverage': '0.00',
      'deductible': '0.00',
      'patient_responsibility': '0.00',
      'paid_amount': '74.00'},
     {'date_of_service': '02/13/26',
      'service_code': 'D0274',
      'billed_amount': '99.00',
      'ppo_discount': '42.00',
      'non_covered': '0.00',
      'other_coverage': '0.00',
      'deductible': '0.00',
      'patient_responsibility': '0.00',
      'paid_amount': '57.00'},
     {'date_of_service': '02/13/26',
      'service_code': 'D0220',
      'billed_amount': '45.00',
      'ppo_discount': '18.00',
      'non_covered': '0.00',
      'ot

: 

## Test 1

In [2]:
import os

folder_path = r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/ANTHEM/Anthem_PDF"

for filename in os.listdir(folder_path):
    if filename.lower().endswith(".pdf"):
        pdf_path = os.path.join(folder_path, filename)

        print(f"\n{'='*80}")
        print(f"Processing: {filename}")
        print(f"{'='*80}")

        try:
            run_pipeline(pdf_path)
            print(f"✅ Completed: {filename}")

        except Exception as e:
            print(f"❌ Failed: {filename}")
            print(f"Error: {e}")


Processing: Pmt_EOP_841817814.pdf
{310.9}
{402.7, 380.1, 357.5, 334.9}
{323.9, 310.9}


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


claim status :not denied
Processing 1/3


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✔ extracted
Processing 2/3


...

✔ extracted
Processing 4/6


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✔ extracted
Processing 5/6


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✔ extracted
Processing 6/6
✔ extracted

🔍 Validation for [SURENDRA PRASAD, SOMESHWARAN]
--------------------------------------------------------------------------------
✅ billed_amount             computed=475.0      | extracted=475.0      MATCH
✅ ppo_discount              computed=203.0      | extracted=203.0      MATCH
✅ non_covered               computed=0.0        | extracted=0.0        MATCH
✅ other_coverage            computed=0.0        | extracted=0.0        MATCH
✅ deductible                computed=0.0        | extracted=0.0        MATCH
✅ paid_amount               computed=272.0      | extracted=272.0      MATCH
✅ total_record_rows         computed=4          | extracted=4          MATCH
--------------------------------------------------------------------------------
✅ [SURENDRA PRASAD, SOMESHWARAN] Validation PASSED


🔍 Validation for [DHANAKOTTI, PRAMILA]
--------------------------------------------------------------------------------
✅ billed_amount             computed=4

In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/ANTHEM/Anthem_PDF/Pmt_EOP_850396697.pdf")

{393.8, 380.8, 406.8, 367.8}
{418.5, 356.8, 369.8, 405.5, 382.9}
{418.5, 356.8, 369.8, 405.5, 382.9}
{393.8, 429.4, 367.8, 406.8, 442.5, 380.8}
{385.6, 323.9, 359.6, 337.0, 372.6, 310.9}
{344.5, 370.5, 357.5}


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


claim status :not denied
Processing 1/6


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✔ extracted
Processing 2/6


...

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✔ extracted
Processing 5/6


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✔ extracted
Processing 6/6
✔ extracted

🔍 Validation for [SURENDRA PRASAD, SOMESHWARAN]
--------------------------------------------------------------------------------
✅ billed_amount             computed=475.0      | extracted=475.0      MATCH
✅ ppo_discount              computed=203.0      | extracted=203.0      MATCH
✅ non_covered               computed=0.0        | extracted=0.0        MATCH
✅ other_coverage            computed=0.0        | extracted=0.0        MATCH
✅ deductible                computed=0.0        | extracted=0.0        MATCH
✅ paid_amount               computed=272.0      | extracted=272.0      MATCH
✅ total_record_rows         computed=4          | extracted=4          MATCH
--------------------------------------------------------------------------------
✅ [SURENDRA PRASAD, SOMESHWARAN] Validation PASSED


🔍 Validation for [DHANAKOTTI, PRAMILA]
--------------------------------------------------------------------------------
✅ billed_amount             computed=4

[{'eob_id': '850396697',
  'file_name': 'Pmt_EOP_850396697.pdf',
  'claim_status': 'not denied',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'SURENDRA PRASAD, SOMESHWARAN',
    'claim_payor': 'Anthem Blue Cross and Blue Shield',
    'dob': '2015-02-20',
    'services': [{'date_of_service': '03/13/26',
      'service_code': 'D1206',
      'billed_amount': '64.00',
      'ppo_discount': '16.00',
      'non_covered': '0.00',
      'other_coverage': '0.00',
      'deductible': '0.00',
      'patient_responsibility': '0.00',
      'paid_amount': '48.00'},
     {'date_of_service': '03/13/26',
      'service_code': 'D0150',
      'billed_amount': '148.00',
      'ppo_discount': '61.00',
      'non_covered': '0.00',
      'other_coverage': '0.00',
      'deductible': '0.00',
      'patient_responsibility': '0.00',
      'paid_amount': '87.00'},
     {'date_of_service': '03/13/26',
      'service_code': 'D0274',
      'billed_amount': '104.00',
      'ppo_discount': '37.00',
  

In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/ANTHEM/Anthem_PDF/Pmt_EOP_850396697.pdf")

{393.8, 380.8, 406.8, 367.8}
{418.5, 356.8, 369.8, 405.5, 382.9}
{418.5, 356.8, 369.8, 405.5, 382.9}
{393.8, 429.4, 367.8, 406.8, 442.5, 380.8}
{385.6, 323.9, 359.6, 337.0, 372.6, 310.9}
{344.5, 370.5, 357.5}


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


claim status :not denied
Processing 1/6


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✔ extracted
Processing 2/6


...

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✔ extracted
Processing 5/6


[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✔ extracted
Processing 6/6
✔ extracted

🔍 Validation for [SURENDRA PRASAD, SOMESHWARAN]
--------------------------------------------------------------------------------
✅ billed_amount             computed=475.0      | extracted=475.0      MATCH
✅ ppo_discount              computed=203.0      | extracted=203.0      MATCH
✅ non_covered               computed=0.0        | extracted=0.0        MATCH
✅ other_coverage            computed=0.0        | extracted=0.0        MATCH
✅ deductible                computed=0.0        | extracted=0.0        MATCH
✅ paid_amount               computed=272.0      | extracted=272.0      MATCH
✅ total_record_rows         computed=4          | extracted=4          MATCH
--------------------------------------------------------------------------------
✅ [SURENDRA PRASAD, SOMESHWARAN] Validation PASSED


🔍 Validation for [DHANAKOTTI, PRAMILA]
--------------------------------------------------------------------------------
✅ billed_amount             computed=0

[{'eob_id': '850396697',
  'file_name': 'Pmt_EOP_850396697.pdf',
  'claim_status': 'not denied',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'SURENDRA PRASAD, SOMESHWARAN',
    'claim_payor': 'Anthem Blue Cross and Blue Shield',
    'dob': '2015-02-20',
    'services': [{'date_of_service': '03/13/26',
      'service_code': 'D1206',
      'billed_amount': '64.00',
      'ppo_discount': '16.00',
      'non_covered': '0.00',
      'other_coverage': '0.00',
      'deductible': '0.00',
      'patient_responsibility': '0.00',
      'paid_amount': '48.00'},
     {'date_of_service': '03/13/26',
      'service_code': 'D0150',
      'billed_amount': '148.00',
      'ppo_discount': '61.00',
      'non_covered': '0.00',
      'other_coverage': '0.00',
      'deductible': '0.00',
      'patient_responsibility': '0.00',
      'paid_amount': '87.00'},
     {'date_of_service': '03/13/26',
      'service_code': 'D0274',
      'billed_amount': '104.00',
      'ppo_discount': '37.00',
  

In [2]:
run_pipeline(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/ANTHEM/Anthem_PDF/Pmt_EOP_837531015.pdf")

[transformers] Kwargs passed to `processor.__call__` have to be in `processor_kwargs` dict, not in `**kwargs`
[transformers] Keyword argument `temperature` is not a valid argument for this processor and will be ignored.
[transformers] Keyword argument `do_sample` is not a valid argument for this processor and will be ignored.


{355.5, 368.5, 404.1, 436.3, 342.4, 381.5}
claim status :not denied
Processing 1/1


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=2500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✔ extracted

🔍 Validation for [JORDAN DEJACO]
--------------------------------------------------------------------------------
✅ billed_amount             computed=1054.0     | extracted=1054.0     MATCH
✅ ppo_discount              computed=383.0      | extracted=383.0      MATCH
✅ non_covered               computed=0.0        | extracted=0.0        MATCH
✅ other_coverage            computed=0.0        | extracted=0.0        MATCH
✅ deductible                computed=118.0      | extracted=118.0      MATCH
✅ paid_amount               computed=553.0      | extracted=553.0      MATCH
✅ total_record_rows         computed=6          | extracted=6          MATCH
--------------------------------------------------------------------------------
✅ [JORDAN DEJACO] Validation PASSED

✅ Success output + pdf + crops saved: EOB_output_success/Anthem_zelis/837531015
Cropped images saved in: EOB_OUTPUT/Zelis/837531015/cropped_images
Success json: EOB_output_success/Anthem_zelis/837531015/837531015_out

[{'eob_id': '837531015',
  'file_name': 'Pmt_EOP_837531015.pdf',
  'claim_status': 'not denied',
  'confidence_score': 100.0,
  'patients': [{'patient_name': 'JORDAN DEJACO',
    'claim_payor': 'UMR',
    'dob': '',
    'services': [{'date_of_service': '02/13/26',
      'service_code': 'D0150',
      'billed_amount': '139.00',
      'ppo_discount': '65.00',
      'non_covered': '0.00',
      'other_coverage': '0.00',
      'deductible': '0.00',
      'patient_responsibility': '0.00',
      'paid_amount': '74.00'},
     {'date_of_service': '02/13/26',
      'service_code': 'D0274',
      'billed_amount': '99.00',
      'ppo_discount': '42.00',
      'non_covered': '0.00',
      'other_coverage': '0.00',
      'deductible': '0.00',
      'patient_responsibility': '0.00',
      'paid_amount': '57.00'},
     {'date_of_service': '02/13/26',
      'service_code': 'D0220',
      'billed_amount': '45.00',
      'ppo_discount': '18.00',
      'non_covered': '0.00',
      'other_coverage': '0.00